Dataset is located within shared google drive. Sign in to access.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os
print(os.path.exists('/content/drive/MyDrive/train'))
print(os.listdir('/content/drive/MyDrive/train'))

True
['Glioblastoma', 'Metastatic', 'Schwannoma', 'glioma_tumor', 'meningioma_tumor', 'no_tumor', 'pituitary_tumor']


In [10]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.nn.init as init
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, Subset
from torchvision import transforms, models
from torchvision.datasets import ImageFolder
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, accuracy_score, f1_score
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import numpy as np
import time
import math
from collections import Counter

torch.manual_seed(73)
np.random.seed(73)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ---------- load dataset ---------- #
# location of the drive in my personal google drive -Bradly
# the way i set this up, is after Ethan shared the drive with me
# right click the folder, organize, add shortcut, then add it onto
# 'myDrive'

# loaded

DATASET_PATH = '/content/drive/MyDrive/train'

train_transforms = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

full_dataset = ImageFolder(root=DATASET_PATH, transform=train_transforms)
N = len(full_dataset)

print(f"Total images:  {len(full_dataset)}")
print(f"Classes found: {full_dataset.classes}")
print(f"Class mapping: {full_dataset.class_to_idx}")

train_idx = [i for i in range(N) if i % 10 not in (0, 1)]
val_idx = list(range(1, N, 10))
test_idx = list(range(0, N, 10))

train_len = len(train_idx)
val_len = len(val_idx)
test_len = len(test_idx)
print(f"Train samples: {train_len}  Val samples: {val_len}  Test samples: {test_len}")

train_ds = Subset(full_dataset, train_idx)
val_ds = Subset(full_dataset, val_idx)
test_ds = Subset(full_dataset, test_idx)

batch_size = 32
num_workers = 4
train_load = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=True, persistent_workers=True)
val_load = DataLoader(val_ds,  batch_size=val_len, shuffle=False, num_workers=max(1, num_workers//2), pin_memory=True, persistent_workers=True)
test_load = DataLoader(test_ds,  batch_size=test_len, shuffle=False, num_workers=max(1, num_workers//2), pin_memory=True, persistent_workers=True)

print("Data Loaded!")

# ---------- set up resnet18 neural network ---------- #
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1) # sets up resnet18 pretrained model

model.fc = nn.Linear(model.fc.in_features, 7) # creates output layer (7 outputs; one for each classification)

init.kaiming_normal_(model.fc.weight, mode='fan_in', nonlinearity='relu')
init.zeros_(model.fc.bias) # initialises output layer weights (randoms for ReLu) and biases (zeros)

#model = model.to(device) # move the model to the GPU

print("Model Initialised!")

# ---------- fine tune model using LoRA rank-4 ---------- # (no need to set up until after midterm report)


class LoRALayer(nn.Module):
    def __init__(self, original_layer, rank=4):
        super().__init__()
        self.original_layer = original_layer
        in_features  = original_layer.in_features
        out_features = original_layer.out_features

        # freeze the original layer
        for param in self.original_layer.parameters():
            param.requires_grad = False

        # LoRA matrices A and B
        self.lora_A = nn.Parameter(torch.randn(in_features, rank) * 0.01)
        self.lora_B = nn.Parameter(torch.zeros(rank, out_features))
        self.rank = rank

    def forward(self, x):
        return self.original_layer(x) + x @ self.lora_A @ self.lora_B

# freeze all backbone layers
for param in model.parameters():
    param.requires_grad = False

# wrap the final fc layer with LoRA rank-4
model.fc = LoRALayer(model.fc, rank=4)
model = model.to(device)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'Trainable parameters: {trainable:,} out of {total:,}')


# ---------- Training loop ---------- #


labels_list = [label for _, label in full_dataset.samples]
class_counts = Counter(labels_list)
total_samples = sum(class_counts.values())
weights = [total_samples / class_counts[i] for i in range(7)]
weights = torch.tensor(weights, dtype=torch.float).to(device)

optimizer = optim.Adam(model.parameters(), lr=0.0002)
criterion = nn.CrossEntropyLoss(weight=weights)

NUM_EPOCHS = 20
best_f1 = 0

print("Starting Training")

for epoch in range(NUM_EPOCHS):
    # train
    model.train()
    running_loss = 0.0
    for i, (images, labels) in enumerate(train_load):
        #if i % 10 == 0: print("iteration: "+str(i))
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    # validate
    #print("Validating...")
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in val_load:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            preds = torch.argmax(outputs, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    macro_f1 = f1_score(all_labels, all_preds, average='macro')
    avg_loss = running_loss / len(train_load) # Calculate average loss per batch
    print(f"Epoch {epoch+1}/{NUM_EPOCHS} | Loss: {avg_loss:.4f} | Macro-F1: {macro_f1:.4f}")

    if macro_f1 > best_f1:
        best_f1 = macro_f1
        torch.save(model.state_dict(), '/content/drive/MyDrive/best_model.pth')
        print(f"  New best model saved! F1: {best_f1:.4f}")


# implement method to save and load trained networks? (depending on how long training takes, we may not need this)


# ---------- test with Macro-F1 value and graphs ---------- #
'''
model.eval() # warm up GPU if available; this code generated with prompt: "For a resnet18 model, write a short program to warm up the GPU"
with torch.no_grad():
    # warm-up iterations
    if device.type == 'cuda':
        for _ in range(3):
            _ = model(torch.randn(8, 3, 224, 224, device=device))


test_set = iter(test_load)
cur_batch, targets = next(test_set) # load the test set for testing
cur_batch = cur_batch.to(device, non_blocking=True)

t = time.time()
print("Starting Testing!")

with torch.no_grad(): # run the test batch through the model
    logits = model(cur_batch)
torch.cuda.synchronize() if device.type == 'cuda' else None

dt = time.time() - t
print("Duration of test computation: ", dt, " sec")

preds = torch.argmax(logits, dim=1) # find the index of the largest output for each image

preds_cpu = preds.cpu().numpy()
targets_cpu = targets.cpu().numpy()

labels = list(range(7))
cm = confusion_matrix(targets_cpu, preds_cpu, labels=labels) # create confusion matrix for viewing

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels) # display confusion matrix
fig, ax = plt.subplots(figsize=(7,7))
disp.plot(ax=ax, cmap=plt.cm.Blues, values_format='d')
plt.title("7x7 Confusion Matrix")
plt.show()

macro_f1 = f1_score(targets_cpu, preds_cpu, labels=list(range(7)), average='macro', zero_division=0) # calculate the Macro-F1 score of the test set and display it
print("Macro F1:", macro_f1)
'''


Using device: cuda


Traceback (most recent call last):
  File "/usr/lib/python3.12/multiprocessing/queues.py", line 259, in _feed
    reader_close()
  File "/usr/lib/python3.12/multiprocessing/connection.py", line 178, in close
    self._close()
  File "/usr/lib/python3.12/multiprocessing/connection.py", line 377, in _close
    _close(self._handle)
OSError: [Errno 9] Bad file descriptor
Traceback (most recent call last):
  File "/usr/lib/python3.12/multiprocessing/queues.py", line 259, in _feed
    reader_close()
  File "/usr/lib/python3.12/multiprocessing/connection.py", line 178, in close
    self._close()
  File "/usr/lib/python3.12/multiprocessing/connection.py", line 377, in _close
    _close(self._handle)
OSError: [Errno 9] Bad file descriptor
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to 

Total images:  7897
Classes found: ['Glioblastoma', 'Metastatic', 'Schwannoma', 'glioma_tumor', 'meningioma_tumor', 'no_tumor', 'pituitary_tumor']
Class mapping: {'Glioblastoma': 0, 'Metastatic': 1, 'Schwannoma': 2, 'glioma_tumor': 3, 'meningioma_tumor': 4, 'no_tumor': 5, 'pituitary_tumor': 6}
Train samples: 6317  Val samples: 790  Test samples: 790
Data Loaded!
Model Initialised!
Trainable parameters: 2,076 out of 11,182,179
Starting Training


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


iteration: 0
iteration: 10
iteration: 20
iteration: 30
iteration: 40
iteration: 50
iteration: 60
iteration: 70
iteration: 80
iteration: 90
iteration: 100
iteration: 110
iteration: 120
iteration: 130
iteration: 140
iteration: 150
iteration: 160
iteration: 170
iteration: 180
iteration: 190
Validating...
Epoch 1/20 | Loss: 2.1488 | Macro-F1: 0.2479
  New best model saved! F1: 0.2479
iteration: 0
iteration: 10
iteration: 20
iteration: 30
iteration: 40
iteration: 50
iteration: 60
iteration: 70
iteration: 80
iteration: 90
iteration: 100
iteration: 110
iteration: 120
iteration: 130
iteration: 140
iteration: 150
iteration: 160
iteration: 170
iteration: 180
iteration: 190
Validating...
Epoch 2/20 | Loss: 1.9232 | Macro-F1: 0.2770
  New best model saved! F1: 0.2770
iteration: 0
iteration: 10
iteration: 20
iteration: 30
iteration: 40
iteration: 50
iteration: 60
iteration: 70
iteration: 80
iteration: 90
iteration: 100
iteration: 110
iteration: 120
iteration: 130
iteration: 140
iteration: 150
itera

'\nmodel.eval() # warm up GPU if available; this code generated with prompt: "For a resnet18 model, write a short program to warm up the GPU"\nwith torch.no_grad():\n    # warm-up iterations\n    if device.type == \'cuda\':\n        for _ in range(3):\n            _ = model(torch.randn(8, 3, 224, 224, device=device))\n\n\ntest_set = iter(test_load)\ncur_batch, targets = next(test_set) # load the test set for testing\ncur_batch = cur_batch.to(device, non_blocking=True)\n\nt = time.time()\nprint("Starting Testing!")\n\nwith torch.no_grad(): # run the test batch through the model\n    logits = model(cur_batch)\ntorch.cuda.synchronize() if device.type == \'cuda\' else None\n\ndt = time.time() - t\nprint("Duration of test computation: ", dt, " sec")\n\npreds = torch.argmax(logits, dim=1) # find the index of the largest output for each image\n\npreds_cpu = preds.cpu().numpy()\ntargets_cpu = targets.cpu().numpy()\n\nlabels = list(range(7))\ncm = confusion_matrix(targets_cpu, preds_cpu, labels